# 피처 선택 파이프라인

## 처리 순서
| 단계 | 방법 | 기준 |
|------|------|------|
| STEP 1 | **Welch t-test** | p < 0.01 — 정상/부실 그룹 간 평균 유의 차이 |
| STEP 2 | **LASSO (LassoCV)** | alpha를 CV로 결정, 계수 ≠ 0 피처 |
| STEP 3 | **교집합** | 두 방법 모두 통과한 피처만 유지 |
| STEP 4 | **VIF 반복 제거** | VIF > 10 피처를 최댓값부터 1개씩 자동 제거 |

## 제외 피처
- 메타 컬럼: 사업자등록번호, 회계년도, 회사명, 종업원, M코드, 빅4감사, 통계청 산업분류
- 라벨 파생: 부실라벨_ICR3년_diff/ratio, 빅4감사_diff/ratio
- 비재무 파생: 통계청 산업분류_diff/ratio/diff_industry/ratio_industry

## 입력 파일
`10,11,12번 [Train].parquet` — 이미 분리된 학습 데이터 (2012~2022년)

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

## 설정
파라미터를 조정해가며 최적 피처 수를 탐색합니다.

- `P_THRESHOLD`: t-test 유의수준 (낮출수록 더 엄격 → 피처 감소)
- `VIF_THRESHOLD`: 다중공선성 제거 기준 (낮출수록 더 엄격 → 피처 감소)

In [2]:
INPUT_FILE    = '10,11,12번 [Train].parquet'
LABEL_COL     = '부실라벨_ICR3년'
P_THRESHOLD   = 0.01   # t-test 유의수준 (0.05 / 0.01)
VIF_THRESHOLD = 10     # VIF 제거 임계값

EXCLUDE_COLS = {
    '사업자등록번호', '회계년도', '회사명', '종업원', 'M코드', '빅4감사',
    '통계청 한국표준산업분류 코드 11차(대분류)',
    '통계청 한국표준산업분류 11차(중분류)',
    '부실라벨_ICR3년', '부실라벨_ICR3년_diff', '부실라벨_ICR3년_ratio',
    '빅4감사_diff', '빅4감사_ratio',
}

NON_FIN_BASE     = {'통계청 한국표준산업분류 코드 11차(대분류)', '통계청 한국표준산업분류 11차(중분류)'}
DERIVED_SUFFIXES = ('_diff', '_ratio', '_diff_industry', '_ratio_industry')

## 데이터 로드 및 피처 컬럼 결정

In [3]:
df = pd.read_parquet(INPUT_FILE)
print(f'[Train 데이터] {len(df):,}행  |  {len(df.columns)}개 컬럼')
print(f'  회계년도: {df["회계년도"].min()} ~ {df["회계년도"].max()}')
print(f'  라벨 분포: 정상 {(df[LABEL_COL]==0).sum():,}개  부실 {(df[LABEL_COL]==1).sum():,}개')

EXCLUDE_DERIVED = {
    f'{base}{suffix}'
    for base in NON_FIN_BASE
    for suffix in DERIVED_SUFFIXES
    if f'{base}{suffix}' in df.columns
}

feature_cols = [
    c for c in df.select_dtypes(include='number').columns
    if c not in EXCLUDE_COLS and c not in EXCLUDE_DERIVED
]
print(f'  후보 피처: {len(feature_cols)}개')

df_clean = df[feature_cols + [LABEL_COL]].dropna()
print(f'  결측치 제거 후: {len(df_clean):,}행')

X = df_clean[feature_cols].copy()
y = df_clean[LABEL_COL].copy()

normal   = df_clean[y == 0]
distress = df_clean[y == 1]
print(f'  정상: {len(normal):,}개  부실: {len(distress):,}개')

if len(normal) < 2 or len(distress) < 2:
    raise ValueError('정상/부실 그룹 중 하나가 2개 미만')

[Train 데이터] 32,268행  |  260개 컬럼
  회계년도: 2012 ~ 2022
  라벨 분포: 정상 31,087개  부실 1,181개
  후보 피처: 251개
  결측치 제거 후: 32,268행
  정상: 31,087개  부실: 1,181개


---
## STEP 1. Welch t-test

정상(0) / 부실(1) 그룹 간 **평균 차이**가 통계적으로 유의한 피처를 추출합니다.

- **Welch t-test**: 두 그룹의 분산이 다를 수 있다고 가정 (`equal_var=False`) → 재무데이터처럼 분산 차이가 큰 경우에 적합
- p < `P_THRESHOLD` 이면 유의한 피처로 선택

In [4]:
ttest_log = []
for col in feature_cols:
    a = normal[col].dropna()
    b = distress[col].dropna()
    if len(a) < 2 or len(b) < 2:
        ttest_log.append({'feature': col, 'p_value': np.nan, 'selected': False, 'reason': '샘플 부족'})
        continue
    _, p = stats.ttest_ind(a, b, equal_var=False)
    ttest_log.append({'feature': col, 'p_value': round(p, 6), 'selected': bool(p < P_THRESHOLD), 'reason': ''})

ttest_df       = pd.DataFrame(ttest_log).sort_values('p_value')
ttest_selected = ttest_df[ttest_df['selected']]['feature'].tolist()
ttest_dropped  = ttest_df[~ttest_df['selected']]['feature'].tolist()

print(f'유의 피처 (p < {P_THRESHOLD}): {len(ttest_selected)}개  |  탈락: {len(ttest_dropped)}개')
print()
ttest_df[ttest_df['selected']][['feature', 'p_value']].reset_index(drop=True)

유의 피처 (p < 0.01): 216개  |  탈락: 35개



,feature,p_value
0,부채비율,0.000000
1,총부채비율,0.000000
2,장기부채비율,0.000000
3,장기부채의존도,0.000000
4,차입금의존도,0.000000
...,...,...
211,재고자산보유기간_ratio,0.006214
212,영업이익증가율_ratio,0.006408
213,재고자산보유기간_diff,0.006965
214,매입채무지급기간,0.007500


---
## STEP 2. LASSO (LassoCV)

**L1 정규화 회귀**로 불필요한 피처의 계수를 0으로 수렴시켜 자동 선택합니다.

- `LassoCV`: 교차검증으로 최적 alpha를 자동 결정
- 계수 ≠ 0 인 피처만 선택
- 표준화(`StandardScaler`) 적용 → 피처 간 스케일 차이 제거

> **alpha 의미**: 클수록 더 강한 정규화 → 더 많은 계수가 0이 됨 (피처 감소)

In [5]:
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

cv_folds = min(5, min(len(normal), len(distress)))
print(f'CV folds: {cv_folds}')

lasso_cv = LassoCV(cv=cv_folds, random_state=42, max_iter=50000, n_jobs=-1)
lasso_cv.fit(X_scaled, y)

lasso_coef     = pd.Series(lasso_cv.coef_, index=feature_cols).rename('coef')
lasso_selected = lasso_coef[lasso_coef != 0].index.tolist()
lasso_dropped  = lasso_coef[lasso_coef == 0].index.tolist()

print(f'선택 alpha: {lasso_cv.alpha_:.6f}')
print(f'계수 ≠ 0: {len(lasso_selected)}개  |  계수 = 0 (탈락): {len(lasso_dropped)}개')
print()

lasso_coef[lasso_coef != 0].abs().sort_values(ascending=False)\
    .reset_index().rename(columns={'index': 'feature', 'coef': '|coef|'})

CV folds: 5
선택 alpha: 0.000142
계수 ≠ 0: 104개  |  계수 = 0 (탈락): 147개



,feature,|coef|
0,영업이익률,0.042179
1,부채비율변화,0.024114
2,유보율_ratio,0.021362
3,영업이익률변화_diff,0.020945
4,부채비율변화_diff,0.019274
...,...,...
99,유형자산비율_ratio,0.000144
100,영업이익증가율_ratio,0.000072
101,감가상각비율,0.000070
102,재고자산보유기간,0.000048


---
## STEP 3. 교집합 (t-test AND LASSO)

두 방법을 **모두** 통과한 피처만 남깁니다.

- t-test는 단변량 유의성, LASSO는 다변량 기여도를 각각 검증
- 교집합을 취함으로써 두 관점에서 모두 의미 있는 피처를 선별

In [6]:
ttest_set    = set(ttest_selected)
lasso_set    = set(lasso_selected)
intersection = sorted(ttest_set & lasso_set)

only_ttest = sorted(ttest_set - lasso_set)
only_lasso = sorted(lasso_set - ttest_set)
neither    = sorted(set(feature_cols) - ttest_set - lasso_set)

print(f't-test 유의:   {len(ttest_selected):>4}개')
print(f'LASSO 선택:    {len(lasso_selected):>4}개')
print(f'교집합:        {len(intersection):>4}개  ← STEP 4로 진행')
print(f't-test만 통과: {len(only_ttest):>4}개  (LASSO 탈락)')
print(f'LASSO만 통과:  {len(only_lasso):>4}개  (t-test 탈락)')
print(f'모두 탈락:     {len(neither):>4}개')
print()
for f in intersection:
    print(f'  {f}')

t-test 유의:    216개
LASSO 선택:     104개
교집합:          84개  ← STEP 4로 진행
t-test만 통과:  132개  (LASSO 탈락)
LASSO만 통과:    20개  (t-test 탈락)
모두 탈락:       15개

  EBITDA마진_diff
  EBITDA증가율
  EBITDA증가율_ratio
  EBIT대매출액_ratio_industry
  FCF
  FCF_diff
  FCF_ratio
  FCF_총자산_diff
  FCF증가율_diff
  ROE
  ROIC_ratio
  감가상각비율
  감가상각비율_ratio
  금융부채비율_diff
  금융비용부담률_diff
  금융비용부담률_ratio
  당좌비율_추정_diff
  매입채무회전율_diff
  매출액순이익률_diff_industry
  매출액증가율_diff
  매출액증가율_ratio
  매출액증가율_ratio_industry
  매출채권회수기간_ratio
  부채비율_ratio_industry
  부채비율변화
  부채비율변화_diff
  부채증가율
  부채증가율_ratio
  비유동비율_diff_industry
  비유동자산회전율_diff_industry
  비유동장기적합률_diff_industry
  순운전자본대총자본_ratio_industry
  순운전자본비율_ratio
  순운전자본회전율
  순운전자본회전율_ratio
  순이익증가율
  순이익증가율_diff
  순차입금비율
  영업CF_유동부채_diff
  영업CF_유동부채_ratio
  영업CF_총부채_diff
  영업이익률
  영업이익률변화_diff
  영업이익증가율
  영업이익증가율_diff
  영업이익증가율_ratio
  영업현금흐름비율
  영업현금흐름비율_ratio
  영업현금흐름증가율_diff
  유동비율_diff
  유보율_diff
  유보율_ratio
  유형자산부채비율_diff
  유형자산비율
  유형자산비율_ratio
  유형자산증가율_diff_industry
  유형자산증가

---
## STEP 4. VIF 반복 제거

**분산팽창지수(VIF)** 로 다중공선성을 제거합니다.

- VIF > `VIF_THRESHOLD` 인 피처 중 가장 높은 것 1개씩 제거
- 반복 후 모든 VIF ≤ `VIF_THRESHOLD` 가 될 때까지 계속

> **VIF 기준**: 10 이상이면 다른 피처로 90% 이상 설명되는 중복 피처
> 1개씩 제거하는 이유: 한 피처를 제거하면 나머지 VIF가 변하기 때문

In [7]:
def calc_vif(X_df: pd.DataFrame) -> pd.DataFrame:
    vif_vals = [
        variance_inflation_factor(X_df.values.astype(float), i)
        for i in range(X_df.shape[1])
    ]
    return (
        pd.DataFrame({'feature': X_df.columns, 'VIF': vif_vals})
        .sort_values('VIF', ascending=False)
        .reset_index(drop=True)
    )

selected        = list(intersection)
vif_removed_log = []
iteration       = 0

while len(selected) >= 2:
    vif_df  = calc_vif(df_clean[selected].copy())
    max_row = vif_df.iloc[0]

    if max_row['VIF'] <= VIF_THRESHOLD:
        print(f'반복 {iteration}: 모든 VIF ≤ {VIF_THRESHOLD} — 종료')
        break

    remove_col = max_row['feature']
    remove_vif = round(max_row['VIF'], 2)
    vif_removed_log.append({'iteration': iteration + 1, 'removed': remove_col, 'VIF': remove_vif})
    selected.remove(remove_col)
    print(f'반복 {iteration + 1}: 제거 [{remove_col}]  VIF={remove_vif}  (남은: {len(selected)}개)')
    iteration += 1
else:
    if len(selected) < 2:
        print('남은 피처 수 < 2 — VIF 계산 불가, 종료')

print(f'\nVIF 제거: {len(vif_removed_log)}개  →  최종 피처: {len(selected)}개')

반복 1: 제거 [투하자본회전율]  VIF=653.61  (남은: 83개)
반복 2: 제거 [영업이익률]  VIF=167.09  (남은: 82개)
반복 3: 제거 [영업CF_유동부채_diff]  VIF=94.93  (남은: 81개)
반복 4: 제거 [유형자산비율]  VIF=78.89  (남은: 80개)
반복 5: 제거 [금융부채비율_diff]  VIF=63.99  (남은: 79개)
반복 6: 제거 [총자산증가율_ratio_industry]  VIF=51.34  (남은: 78개)
반복 7: 제거 [금융비용부담률_ratio]  VIF=45.95  (남은: 77개)
반복 8: 제거 [감가상각비율]  VIF=36.4  (남은: 76개)
반복 9: 제거 [유보율_diff]  VIF=33.42  (남은: 75개)
반복 10: 제거 [부채비율변화_diff]  VIF=25.09  (남은: 74개)
반복 11: 제거 [현금ROA]  VIF=24.84  (남은: 73개)
반복 12: 제거 [자기자본증가율_ratio_industry]  VIF=19.97  (남은: 72개)
반복 13: 제거 [EBIT대매출액_ratio_industry]  VIF=18.78  (남은: 71개)
반복 14: 제거 [비유동비율_diff_industry]  VIF=18.39  (남은: 70개)
반복 15: 제거 [영업CF_유동부채_ratio]  VIF=18.25  (남은: 69개)
반복 16: 제거 [유동비율_diff]  VIF=17.86  (남은: 68개)
반복 17: 제거 [재고자산회전율_ratio]  VIF=16.92  (남은: 67개)
반복 18: 제거 [총자산증가율_diff_industry]  VIF=15.06  (남은: 66개)
반복 19: 제거 [자기자본비율_ratio]  VIF=13.48  (남은: 65개)
반복 20: 제거 [유보율_ratio]  VIF=13.34  (남은: 64개)
반복 21: 제거 [매출액증가율_ratio_industry]  VIF=13.06  (남은: 63개)
반복 

---
## 최종 결과

In [8]:
print(f'요약: {len(feature_cols)}개 → t-test {len(ttest_selected)}개 → 교집합 {len(intersection)}개 → 최종 {len(selected)}개')
print(f'EPV (부실 샘플 수 / 최종 피처 수) = {len(distress)} / {len(selected)} = {len(distress)/len(selected):.1f}')
print()
for i, f in enumerate(selected, 1):
    print(f'  {i:>3}. {f}')

요약: 251개 → t-test 216개 → 교집합 84개 → 최종 60개
EPV (부실 샘플 수 / 최종 피처 수) = 1181 / 60 = 19.7

    1. EBITDA마진_diff
    2. EBITDA증가율
    3. EBITDA증가율_ratio
    4. FCF
    5. FCF_diff
    6. FCF_ratio
    7. FCF_총자산_diff
    8. FCF증가율_diff
    9. ROE
   10. ROIC_ratio
   11. 감가상각비율_ratio
   12. 금융비용부담률_diff
   13. 당좌비율_추정_diff
   14. 매입채무회전율_diff
   15. 매출액순이익률_diff_industry
   16. 매출액증가율_diff
   17. 매출액증가율_ratio
   18. 매출채권회수기간_ratio
   19. 부채비율_ratio_industry
   20. 부채비율변화
   21. 부채증가율
   22. 부채증가율_ratio
   23. 비유동자산회전율_diff_industry
   24. 비유동장기적합률_diff_industry
   25. 순운전자본비율_ratio
   26. 순운전자본회전율
   27. 순이익증가율
   28. 순이익증가율_diff
   29. 순차입금비율
   30. 영업CF_총부채_diff
   31. 영업이익률변화_diff
   32. 영업이익증가율
   33. 영업이익증가율_diff
   34. 영업이익증가율_ratio
   35. 영업현금흐름비율
   36. 영업현금흐름비율_ratio
   37. 영업현금흐름증가율_diff
   38. 유형자산부채비율_diff
   39. 유형자산비율_ratio
   40. 유형자산증가율_diff_industry
   41. 유형자산증가율_ratio
   42. 유형자산회전율_ratio
   43. 자기자본증가율_diff_industry
   44. 자기자본증가율_ratio
   45. 자기자본회전율_diff_industry
   46.

### 최종 피처 VIF 확인

In [9]:
calc_vif(df_clean[selected].copy())

,feature,VIF
0,장기부채비율_ratio,9.802321
1,총자산회전율,7.732167
2,순차입금비율,7.585837
3,자기자본증가율_diff_industry,7.113197
4,장기부채의존도_ratio,6.995769
5,유형자산비율_ratio,6.841079
6,부채비율_ratio_industry,6.813547
7,영업이익증가율,6.633086
8,자기자본증가율_ratio,6.476348
9,비유동자산회전율_diff_industry,6.455771


---
## 저장
- `*_피처선택로그.parquet` : 전체 피처별 단계별 탈락 이유
- `*_최종피처.parquet` : 최종 선택된 피처 목록

In [14]:
rows = []
for f in feature_cols:
    in_ttest = f in ttest_set
    in_lasso = f in lasso_set
    vif_info = next((r for r in vif_removed_log if r['removed'] == f), None)

    if f in selected:
        stage = '최종 선택'
    elif vif_info:
        stage = f'STEP4_VIF제거 (VIF={vif_info["VIF"]}, 반복{vif_info["iteration"]})'
    elif f in set(intersection):
        stage = '알 수 없음'
    elif in_ttest and not in_lasso:
        stage = 'STEP3_LASSO미통과'
    elif in_lasso and not in_ttest:
        stage = 'STEP3_ttest미통과'
    else:
        stage = 'STEP1·STEP2_모두탈락'

    p_val = next((r['p_value'] for r in ttest_log if r['feature'] == f), np.nan)
    rows.append({
        'feature':        f,
        'ttest_p':        p_val,
        'ttest_pass':     in_ttest,
        'lasso_coef':     round(float(lasso_coef.get(f, 0)), 6),
        'lasso_pass':     in_lasso,
        'intersection':   f in set(intersection),
        'final_selected': f in selected,
        'stage':          stage,
    })



log_path   = f'13번. 피처선택로그.parquet'
final_path = f'13번. 최종피처.parquet'

pd.DataFrame(rows).to_parquet(
    log_path,
    index=False
)

pd.DataFrame({'final_feature': selected}).to_parquet(
    final_path,
    index=False
)

print(f'저장 완료: {log_path}')
print(f'저장 완료: {final_path}')

저장 완료: 13번. 피처선택로그.parquet
저장 완료: 13번. 최종피처.parquet
